<a href="https://colab.research.google.com/github/Longhanhmid24/DoAn_Email/blob/main/T5_strep6_7_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Sử dụng mô hình Text-to-Text Transfer Transformer

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

base_path = "/content/drive/MyDrive/Deep_Learning_Email/dataset_split/"


train_df = pd.read_csv(base_path + "train.csv")
val_df   = pd.read_csv(base_path + "val.csv")
test_df  = pd.read_csv(base_path + "test.csv")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import numpy as np
import torch
import math
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
import optuna
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

SETUP VÀ TẢI DỮ LIỆU

In [11]:
# Chuyển pandas dataframe sang dạng Dataset của HuggingFace để xử lý cho nhanh
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Khởi tạo Tokenizer của T5-small (nhẹ, train nhanh trên máy cá nhân)
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Hàm chuyển đổi văn bản thành Input IDs (Tokenization)
def preprocess_function(examples):
    # Prefix "reply email:" giúp T5 biết nó đang làm nhiệm vụ gì
    inputs = ["reply email: " + doc for doc in examples["Input_Text"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Xử lý nhãn (Output_Text)
    labels = tokenizer(examples["Output_Text"], max_length=256, truncation=True, padding="max_length")
    # Thay thế id của pad token bằng -100 để mô hình bỏ qua khi tính Loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Đang mã hóa Token (Tokenization)...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

# Công cụ tự động gộp batch dữ liệu
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đang mã hóa Token (Tokenization)...


Map:   0%|          | 0/11069 [00:00<?, ? examples/s]

Map:   0%|          | 0/1384 [00:00<?, ? examples/s]

Map:   0%|          | 0/1384 [00:00<?, ? examples/s]